# 국가핵심기술 키워드 ↔ 표준 용어 유사도 분석

**목표**: 국가핵심기술 키워드와 의미가 비슷한 표준 용어(`df_word`)를 임베딩 유사도로 찾는다.

**진행 순서**
1. 설정 및 데이터 불러오기
2. 키워드 리스트 만들기 (`L5` + `keyword` 합치기 → `,` 기준으로 쪼개기)
3. 사내 LLM API로 임베딩 생성
4. 코사인 유사도 계산
5. 결과 확인 및 저장

## 1. 설정 및 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [ ]:
import os

# 사내 LLM API 설정 (주소와 키는 환경변수로 설정)
BASE_URL = os.environ.get("LLM_BASE_URL", "http://<사내 LLM 주소>/v1")
EMBEDDING_URL = f"{BASE_URL}/embeddings"

OPEN_API_KEY = os.environ.get("LLM_API_KEY", "<사내 API 키>")   # 사내 embeddings API 구독 필요
MODEL_NAME = "bge-m3"

HEADERS = {
    "Authorization": f"Bearer {OPEN_API_KEY}",
    "Content-Type": "application/json",
}

In [ ]:
# 데이터 불러오기 (파일 경로/형식은 실제에 맞게 수정)
df_keyword = pd.read_excel("국가핵심기술_키워드.xlsx")   # 컬럼: L5, keyword
df_word = pd.read_excel("표준용어.xlsx")                 # 컬럼: 설명 포함

print("df_keyword:", df_keyword.shape)
print("df_word   :", df_word.shape)

In [ ]:
df_keyword.head()

In [ ]:
df_word.head()

## 2. 키워드 리스트 만들기

- `L5` 열과 `keyword` 열의 값을 하나의 열로 합친다.
- 한 칸에 `,`로 여러 값이 들어 있으면 각각 쪼갠다.
- 앞뒤 공백 제거, 빈 값·중복 제거
- 나중에 추적할 수 있도록 **어느 L5에서 나온 키워드인지**도 함께 남겨둔다.

In [ ]:
# L5 값 자체도 키워드로 사용 → (L5, 값) 형태로 세로로 쌓기
df_from_l5 = df_keyword[["L5"]].copy()
df_from_l5["keyword_raw"] = df_keyword["L5"]

df_from_kw = df_keyword[["L5"]].copy()
df_from_kw["keyword_raw"] = df_keyword["keyword"]

df_kw_long = pd.concat([df_from_l5, df_from_kw], ignore_index=True)
df_kw_long = df_kw_long.dropna(subset=["keyword_raw"])

print("합친 후 행 수:", len(df_kw_long))
df_kw_long.head()

In [ ]:
# ',' 기준으로 쪼개서 한 행에 하나의 키워드만 남기기
df_kw_long["keyword"] = df_kw_long["keyword_raw"].astype(str).str.split(",")
df_kw_long = df_kw_long.explode("keyword")

# 공백 제거 후 빈 문자열 제거
df_kw_long["keyword"] = df_kw_long["keyword"].str.strip()
df_kw_long = df_kw_long[df_kw_long["keyword"] != ""]

df_kw_long = df_kw_long[["L5", "keyword"]].drop_duplicates().reset_index(drop=True)

print("쪼갠 후 행 수:", len(df_kw_long))
df_kw_long.head(20)

In [ ]:
# 임베딩할 고유 키워드 리스트 (같은 키워드가 여러 L5에 걸쳐 있을 수 있으므로 중복 제거)
keyword_list = df_kw_long["keyword"].drop_duplicates().tolist()

print("고유 키워드 수:", len(keyword_list))
keyword_list[:20]

## 3. 임베딩 생성

- 한 번에 여러 문장을 보내 호출 횟수를 줄인다. (`batch_size`개씩)
- 429(Too Many Requests)가 오면 잠시 기다렸다가 재시도하고, 배치 사이에 `sleep_sec`만큼 쉰다.
- API 호출은 시간이 걸리므로 결과를 `.npy` 파일로 저장해 두고 재사용한다.

In [ ]:
import time

def get_embeddings(texts, batch_size=16, sleep_sec=0.5, max_retry=5):
    """texts 리스트를 batch_size개씩 임베딩. 429가 오면 기다렸다가 다시 시도"""
    all_vectors = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        body = {"input": batch, "model": MODEL_NAME}

        for attempt in range(max_retry):
            rsp = requests.post(EMBEDDING_URL, headers=HEADERS, json=body, timeout=60)

            if rsp.status_code == 429:
                # 서버가 기다릴 시간을 알려주면 그만큼, 아니면 2, 4, 8, ...초씩 늘려가며 대기
                wait = int(rsp.headers.get("Retry-After", 2 ** (attempt + 1)))
                print(f"429 발생 → {wait}초 대기 후 재시도 ({attempt + 1}/{max_retry})")
                time.sleep(wait)
                continue

            rsp.raise_for_status()
            break
        else:
            raise RuntimeError(f"{max_retry}번 재시도했지만 계속 429: batch 시작 위치 {i}")

        data = sorted(rsp.json()["data"], key=lambda x: x["index"])
        all_vectors.extend([d["embedding"] for d in data])

        time.sleep(sleep_sec)   # 배치 사이에 쉬어서 한도 초과를 예방

    return np.array(all_vectors)

In [ ]:
# API 연결 테스트
test_vec = get_embeddings(["hello"])
print("임베딩 차원:", test_vec.shape)

In [ ]:
# 키워드 임베딩
keyword_emb = get_embeddings(keyword_list)
np.save("keyword_emb.npy", keyword_emb)

print(keyword_emb.shape)

In [ ]:
# 표준 용어 '설명' 임베딩 (설명이 비어 있는 행은 제외)
df_word_valid = df_word[df_word["설명"].notna()].copy()
df_word_valid["설명"] = df_word_valid["설명"].astype(str).str.strip()
df_word_valid = df_word_valid[df_word_valid["설명"] != ""].reset_index(drop=True)

print("전체 표준 용어 수   :", len(df_word))
print("설명 있는 용어 수   :", len(df_word_valid))

desc_list = df_word_valid["설명"].tolist()
desc_emb = get_embeddings(desc_list)
np.save("desc_emb.npy", desc_emb)

print(desc_emb.shape)

### (커널 재시작 시) 임베딩 다시 돌리지 않고 불러오기

셀 1~2절(키워드 리스트 만들기)까지 실행한 뒤, 위 임베딩 셀들은 건너뛰고 이 셀을 실행한다.

In [ ]:
# 임베딩할 때와 똑같은 순서로 df_word_valid 다시 만들기 (API 호출 없음)
df_word_valid = df_word[df_word["설명"].notna()].copy()
df_word_valid["설명"] = df_word_valid["설명"].astype(str).str.strip()
df_word_valid = df_word_valid[df_word_valid["설명"] != ""].reset_index(drop=True)

# 저장해 둔 임베딩 불러오기
keyword_emb = np.load("keyword_emb.npy")
desc_emb = np.load("desc_emb.npy")

# 순서와 개수가 맞는지 확인 (다르면 데이터 파일이 바뀐 것이니 임베딩을 다시 해야 함)
assert len(keyword_emb) == len(keyword_list), "키워드 개수가 임베딩과 다름"
assert len(desc_emb) == len(df_word_valid), "설명 개수가 임베딩과 다름"
print("임베딩 불러오기 완료:", keyword_emb.shape, desc_emb.shape)

## 4. 코사인 유사도 계산

벡터 길이를 1로 맞춘(정규화) 뒤 행렬곱을 하면 코사인 유사도가 된다.
결과 `sim`의 모양은 **(키워드 수, 설명 수)**.

In [ ]:
def normalize(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

sim = normalize(keyword_emb) @ normalize(desc_emb).T

print("유사도 행렬 shape:", sim.shape)
pd.Series(sim.flatten()).describe(percentiles=[0.5, 0.9, 0.99, 0.999])

## 5. 최종 결과표

- `keyword`: L5 + keyword 합집합 리스트의 키워드
- 매칭된 `df_word`의 모든 열
- `similarity`: 임베딩 유사도

유사도가 `THRESHOLD` 이상인 (키워드, 설명) 쌍을 모두 남긴다.

In [ ]:
THRESHOLD = 0.5

# 유사도가 임계값 이상인 (키워드 번호, 설명 번호) 쌍을 모두 찾기
kw_idx, word_idx = np.where(sim >= THRESHOLD)

df_final = df_word_valid.iloc[word_idx].reset_index(drop=True)          # 매칭된 df_word 행 전체
df_final.insert(0, "keyword", [keyword_list[k] for k in kw_idx])        # 맨 앞에 키워드
df_final["similarity"] = sim[kw_idx, word_idx].round(4)                 # 맨 뒤에 유사도

df_final = df_final.sort_values(["keyword", "similarity"], ascending=[True, False]).reset_index(drop=True)

print(f"매칭 건수: {len(df_final)}")
print(f"매칭된 키워드 수: {df_final['keyword'].nunique()} / {len(keyword_list)}")
df_final.head(50)

In [ ]:
# 키워드별 매칭 건수
df_final["keyword"].value_counts()

## 6. 결과 저장

In [ ]:
df_final.to_excel("국가핵심기술_표준용어_매칭결과.xlsx", index=False)
print("저장 완료")